# Master Verification

Confirms both tables are populated correctly and
each controlled batch has the expected quality issue
at the expected severity level.

In [0]:
from datetime import date, timedelta

TODAY        = date.today()
dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG      = dbutils.widgets.get("catalog")
SCHEMA       = dbutils.widgets.get("schema")
TXN_TABLE    = f"{CATALOG}.{SCHEMA}.uc1_raw_store_transactions"
INV_TABLE    = f"{CATALOG}.{SCHEMA}.uc1_raw_inventory_movements"

print("=" * 70)
print("MASTER VERIFICATION — AGENT 1 PILOT DATA")
print("=" * 70)

## Table 1: Store Transactions Summary

In [0]:
spark.sql(f"""
    SELECT
        CASE
            WHEN transaction_date BETWEEN '{TODAY + timedelta(days=-37)}'
                                  AND     '{TODAY + timedelta(days=-7)}'
            THEN 'Historical (clean)'
            WHEN transaction_date = '{TODAY + timedelta(days=-6)}'
            THEN 'Batch 2: Clean live'
            WHEN transaction_date = '{TODAY + timedelta(days=-5)}'
            THEN 'Batch 3: Volume drop'
            WHEN transaction_date = '{TODAY + timedelta(days=-4)}'
            THEN 'Batch 4: Null spike'
            WHEN transaction_date = '{TODAY + timedelta(days=-3)}'
            THEN 'Batch 5: Categorical'
            WHEN transaction_date = '{TODAY + timedelta(days=-2)}'
            THEN 'Batch 6: Cross-column'
            WHEN transaction_date = '{TODAY + timedelta(days=-1)}'
            THEN 'Batch 7: Combined'
            ELSE 'Other'
        END                                                     AS batch_label,
        COUNT(*)                                                AS row_count,
        ROUND(SUM(CASE WHEN net_amount IS NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)      AS pct_null_net_amount,
        ROUND(SUM(CASE WHEN store_id IS NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)      AS pct_null_store_id,
        COUNT(DISTINCT payment_method)                          AS distinct_payment_methods
    FROM {TXN_TABLE}
    GROUP BY 1
    ORDER BY MIN(transaction_date)
""").show(truncate=False)

## Table 2: Inventory Movements Summary

In [0]:
spark.sql(f"""
    SELECT
        CASE
            WHEN business_date BETWEEN '{TODAY + timedelta(days=-37)}'
                               AND     '{TODAY + timedelta(days=-7)}'
            THEN 'Historical (clean)'
            WHEN business_date = '{TODAY + timedelta(days=-6)}'
            THEN 'Batch 2: Clean live'
            WHEN business_date = '{TODAY + timedelta(days=-5)}'
            THEN 'Batch 3: Malformed'
            WHEN business_date = '{TODAY + timedelta(days=-4)}'
            THEN 'Batch 4: Qty consistency'
            WHEN business_date = '{TODAY + timedelta(days=-3)}'
            THEN 'Batch 5: Cardinality drift'
            WHEN business_date = '{TODAY + timedelta(days=-2)}'
            THEN 'Batch 6: Negative qty'
            WHEN business_date = '{TODAY + timedelta(days=-1)}'
            THEN 'Batch 7: Combined'
            ELSE 'Other'
        END                                                      AS batch_label,
        COUNT(*)                                                 AS event_count,
        COUNT(DISTINCT sku)                                      AS distinct_skus,
        ROUND(SUM(CASE WHEN event_id IS NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)       AS pct_null_event_id,
        ROUND(SUM(CASE WHEN _rescued_data IS NOT NULL
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)       AS pct_malformed,
        ROUND(SUM(CASE WHEN quantity_after < 0
                  THEN 1 ELSE 0 END) / COUNT(*) * 100, 2)       AS pct_negative_qty
    FROM {INV_TABLE}
    GROUP BY 1
    ORDER BY MIN(business_date)
""").show(truncate=False)

## Expected vs Observed Quality Issues

In [0]:
print("""
EXPECTED DETECTION SUMMARY
══════════════════════════════════════════════════════════════════════

raw_store_transactions:

  Batch 2  Clean live         All checks PASS
  Batch 3  Volume drop        VOLUME_ANOMALY → CRITICAL (~77% drop)
  Batch 4  Null spike         NULL_SPIKE net_amount → CRITICAL (22%)
  Batch 5  Categorical        CATEGORICAL_DOMAIN payment_method → WARNING
  Batch 6  Cross-column       CROSS_COLUMN_CONSISTENCY → WARNING (12%)
  Batch 7  Combined           VOLUME_ANOMALY → WARNING
                              NULL_SPIKE store_id → WARNING
                              CATEGORICAL_DOMAIN txn_type → WARNING
                              AI invocation → TRIGGERED

raw_inventory_movements:

  Batch 2  Clean live         All checks PASS
  Batch 3  Malformed          MALFORMED_RECORDS → CRITICAL (18%)
  Batch 4  Qty consistency    CROSS_COLUMN_CONSISTENCY → WARNING (9%)
  Batch 5  Cardinality        CARDINALITY_DRIFT sku → CRITICAL
  Batch 6  Negative qty       NUMERIC_RANGE quantity_after → CRITICAL (4%)
  Batch 7  Combined           VOLUME_ANOMALY → WARNING
                              MALFORMED_RECORDS → WARNING (3%)
                              NULL_SPIKE event_id → CRITICAL (2%)
                              AI invocation → TRIGGERED

══════════════════════════════════════════════════════════════════════
""")

print("Master verification complete.")
print("Data is ready for Agent 1 validation notebook development.")